In [1]:
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords


# Downloading NLTK data
nltk.download('stopwords')   # Downloading stopwords data
nltk.download('punkt')   
test = pd.read_csv("test.csv")
train = pd.read_csv("train.csv")


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/aayushi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/aayushi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


# First look into the data

In [2]:
summary = pd.DataFrame({
    "unique values": train.nunique(),
    "null": train.isnull().sum(),
    "%null": (train.isnull().sum()/len(train)) * 100,
    "dtypes": train.dtypes
})
print(summary)
print("="*100)
for column in train.columns:
    if train[column].nunique() <= 10:
        print(f"Type of {column} values in the Dataset")
        print(train[column].unique())
        print()

print("="*100)
num_col = train.select_dtypes("int").columns
print(train[num_col].describe())

print("="*100)
print("Target column analysis")
print(pd.DataFrame({
    "value count": train["label"].value_counts(),
    "% value": ((train["label"].value_counts()/len(train)) * 100).round(2)
}).to_string())

              unique values    null      %null dtypes
created_date         197996       0   0.000000    str
post_id                  52       0   0.000000  int64
emoticon_1               36       0   0.000000  int64
emoticon_2               10       0   0.000000  int64
emoticon_3               16       0   0.000000  int64
upvote                  122       0   0.000000  int64
downvote                 62       0   0.000000  int64
if_1                     57       0   0.000000  int64
if_2                     81       0   0.000000  int64
race                      6  145423  73.445960    str
religion                  8  145423  73.445960    str
gender                    5  145423  73.445960    str
disability                2       0   0.000000   bool
comment              197842       1   0.000505    str
label                     4       0   0.000000  int64
Type of emoticon_2 values in the Dataset
[ 0  1  2  4  3  5  8  6 11  7]

Type of race values in the Dataset
<StringArray>
[nan, 'none',

# Data cleaning

In [3]:
train_clean = train.copy() 
test_clean = test.copy()
num_col = test_clean.select_dtypes(include=["int"]).columns.tolist()

print("="*100)

print("Outliers summary")
outliers_summary_before_handling = []
outliers_summary_after_handling = []

def handle_outliers(df, col, lower, upper):
    df[col] = np.where(df[col] > upper, upper, np.where(df[col] < lower, lower, df[col]))
    
for col in num_col:
    if col == "post_id":
        continue
    q1 = train_clean[col].quantile(0.25)
    q3 = train_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = train_clean[(train_clean[col] < lower_bound) | (train_clean[col] > upper_bound)]
    outlier_count = len(outliers)
    outlier_percentage = ((outlier_count/len(train_clean)) * 100)

    outliers_summary_before_handling.append({
        "column": col,
        "outlier_count": outlier_count,
        "outlier_percentage": f"{outlier_percentage:.2f}%",
        "lower_bound": lower_bound,
        "upper_bound": upper_bound
    })

    handle_outliers(train_clean, col, lower_bound, upper_bound)
    handle_outliers(test_clean, col, lower_bound, upper_bound)

    outliers = train_clean[(train_clean[col] < lower_bound) | (train_clean[col] > upper_bound)]
    outlier_count = len(outliers)
    outlier_percentage = ((outlier_count/len(train_clean)) * 100)

    outliers_summary_after_handling.append({
        "column": col,
        "outlier_count": outlier_count,
        "outlier_percentage": f"{outlier_percentage:.2f}%",
        "lower_bound": lower_bound,
        "upper_bound": upper_bound
    })
    
print("Outliers Summary before handling")
print(pd.DataFrame(outliers_summary_before_handling).to_string(index=False))

print("="*100)

print("Outliers Summary after handling")
print(pd.DataFrame(outliers_summary_after_handling).to_string(index=False))

print("="*100)


print("Missing values Summary")
cat_col = train_clean.select_dtypes(include=["object"]).columns.tolist()
print((train_clean[cat_col].isnull().sum()/len(train_clean[cat_col]) * 100).round(2))
print("="*100)

for col in cat_col:
    if col == "created_date":
        continue
    train_clean[col] = train_clean[col].fillna("none")
    test_clean[col] = test_clean[col].fillna("none")

print("="*100)

print("Missing values Summary after cleanup")
cat_col = train_clean.select_dtypes(include=["object"]).columns.tolist()
print((train_clean[cat_col].isnull().sum()/len(train_clean[cat_col]) * 100).round(2))

Outliers summary
Outliers Summary before handling
    column  outlier_count outlier_percentage  lower_bound  upper_bound
emoticon_1          28922             14.61%          0.0          0.0
emoticon_2           8109              4.10%          0.0          0.0
emoticon_3          17165              8.67%          0.0          0.0
    upvote          17304              8.74%         -4.5          7.5
  downvote          15173              7.66%         -1.5          2.5
      if_1             85              0.04%         -6.0         10.0
      if_2           3930              1.98%         -5.0         19.0
Outliers Summary after handling
    column  outlier_count outlier_percentage  lower_bound  upper_bound
emoticon_1              0              0.00%          0.0          0.0
emoticon_2              0              0.00%          0.0          0.0
emoticon_3              0              0.00%          0.0          0.0
    upvote              0              0.00%         -4.5         

/tmp/ipykernel_28776/2931641179.py:61: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_col = train_clean.select_dtypes(include=["object"]).columns.tolist()
/tmp/ipykernel_28776/2931641179.py:74: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migrat

# Text preprocessing and feature engineering

In [4]:
from nltk.stem.porter import PorterStemmer
import string
import nltk
from nltk.corpus import stopwords


# Downloading NLTK data
nltk.download('stopwords')   # Downloading stopwords data
nltk.download('punkt')
nltk.download('punkt_tab')


ps = PorterStemmer()

def transform_text(text):

    text = text.lower()

    text = nltk.word_tokenize(text)

    refined_text = []
    for i in text:
        if i.isalnum():
            refined_text.append(i)

    text = refined_text[:]
    refined_text.clear()

    for i in text:
        if i not in stopwords.words("english") and i not in string.punctuation:
            refined_text.append(i)

    text = refined_text[:]
    refined_text.clear()

    for i in text:
        refined_text.append(ps.stem(i))
        
    return " ".join(refined_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/aayushi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/aayushi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/aayushi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [19]:
transform_text('Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...')

'go jurong point crazi avail bugi n great world la e buffet cine got amor wat'

In [5]:
train_featured = train_clean.copy()
test_featured = test_clean.copy()

def extract_text_features(df):
    #df["transformed_comments"] = train_featured["comment"].apply(transform_text)
    df["num_char"] = train_featured["comment"].apply(len)
    df["num_words"] = train_featured["comment"].apply(lambda x: len(nltk.word_tokenize(x)))
    df["num_sentence"] = train_featured["comment"].apply(lambda x: len(nltk.sent_tokenize(x)))
    df["num_caps"] = train["comment"].apply(lambda x: sum(1 for c in x if c.isupper()))
    df["num_exclamation"] = train["comment"].str.count("!")
    df["num_question"] = train["comment"].str.count("\?")

def extract_date(df):
    df["created_date"] = pd.to_datetime(df["created_date"])
    df["day"] = df["created_date"].dt.day
    df["month"] = df["created_date"].dt.month
    df["year"] = df["created_date"].dt.year
    df["is_weekend"] = (df["created_date"].dt.weekday >= 5).astype(int)

def extract_features(df):
    df["net_score"] = df["upvote"] - df["downvote"]
    df["engangement"] = df["upvote"] + df["downvote"]
    #df["votes_ratio"] = df["upvote"]/(df["upvote"] + df["downvote"])

extract_text_features(train_featured)
extract_text_features(test_featured)
extract_features(train_featured)
extract_features(test_featured)
extract_date(train_featured)
extract_date(test_featured)
train_featured["disability"] = np.where(train_featured["disability"] == True, 1, 0)
test_featured["disability"] = np.where(test_featured["disability"] == True, 1, 0)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score

In [7]:
target = train_featured["label"]
train = train_featured.drop(["created_date", "label", "post_id", "upvote", "downvote"], axis=1)
#train = train_featured[["transformed_comments", "num_char", "engangment", "day"]]
test = test_featured.drop(["created_date"], axis=1)
X_train, X_val, y_train, y_val = train_test_split(train, target, test_size=0.20, random_state=42)

In [8]:
cat_col = ["religion", "gender", "race"]
#cat_col = train.select_dtypes(include=["object"]).columns.tolist()
num_col = train.select_dtypes(include=["float64","int64"]).columns.to_list()

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_col),
    ("num_scalar", StandardScaler(), num_col),
    ("vectorizer", TfidfVectorizer(
                lowercase=True,
                stop_words=None,
                ngram_range=(1,2),
                max_features=50000,
                min_df=3,
                max_df=0.9
            ), "comment")
], remainder="passthrough")

preprocessor1 = ColumnTransformer([
    ("vectorizer", TfidfVectorizer(max_features=3000), "comment")
], remainder="passthrough")

In [ ]:
knc = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth = 5)
lrc = LogisticRegression(C=4, max_iter=2000, class_weight="balanced")
rfc = RandomForestClassifier(n_estimators = 400, random_state = 2 )
abc = AdaBoostClassifier(n_estimators = 50, random_state = 2)
bc = BaggingClassifier(n_estimators = 50, random_state = 2)
etc = ExtraTreesClassifier(n_estimators = 50, random_state = 2)
gbdt = GradientBoostingClassifier(n_estimators = 50, random_state = 2)    
xgb  = XGBClassifier(n_estimators = 50, random_state = 2)
lsvc = LinearSVC(class_weight="balanced", max_iter=2000, C=4)
lgbm = model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    class_weight="balanced",
    objective='multiclass',
    num_class=4,
    boosting_type='gbdt',
    metric='multi_logloss',
    random_state=42,
    verbosity=-1
)

clfs = {
    'KNN': knc,
    'NB': mnb,
    'DT': dtc,
    'LR': lrc,
    'RF': rfc,
    'Adaboost': abc,
    'Bgc': bc,
    'ETC': etc,
    'GBDT': gbdt,
    'xgb': xgb
    
}
new_clfs = {
    'LR': lrc,
    'RF': rfc,
    'LSVC': lsvc,
    'Adaboost': abc,
    "LGBM": lgbm
}

models_trained = {}
scores = {}

for name, model in new_clfs.items():
    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])
    pipeline.fit(X_train, y_train)
    models_trained[name] = pipeline
    y_pred = pipeline.predict(X_val)
    f1 = f1_score(y_val, y_pred, average="macro")
    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, average="macro")
    scores[name] = {"f1_score": f1, "accuracy": accuracy, "precision": precision}
    print()
    print("For: ", name)
    print("F1 Score: ", f1)
    print("Accuracy: ", accuracy)
    print("Precision: ", precision)
    

/home/aayushi/Documents/codes/jupyter/ml_algo_practice/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



For:  LR
F1 Score:  0.6039699535852914
Accuracy:  0.7630303030303031
Precision:  0.6364368793972316

For:  RF
F1 Score:  0.7218274706192711
Accuracy:  0.8878282828282829
Precision:  0.8372432252221963

For:  LSVC
F1 Score:  0.7688251980998635
Accuracy:  0.8882323232323233
Precision:  0.7960383113673664
